In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.data.splitter import Splitter
from src.models.baselines import ApolloPipeline

## 1. Configuration & Setup

In [1]:
# Paths
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_DIR = "../data/splits"
SPLIT_FILE = f"{SPLIT_DIR}/manual_splits.json"

# Ensure Splits Exist
if not os.path.exists(SPLIT_FILE):
    print("Generating Manual Splits from Apollo Paper...")
    splitter = Splitter(DATA_PATH, SPLIT_DIR)
    splitter.generate_manual_split()
else:
    print(f"Loading frozen splits from: {SPLIT_FILE}")

import os
import joblib  # Standard tool for saving scikit-learn models
from src.models.baselines import ApolloPipeline  # <--- The new class we made

# Create a directory to store the trained models
MODEL_DIR = "../models/baselines"
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"📂 Model directory ready: {MODEL_DIR}")

NameError: name 'os' is not defined

## 2. Load Data
We use `memory_mapped=True` to keep RAM usage low, although the loader handles this gracefully.

In [3]:
print("Initializing Loader...")
# The loader automatically handles the manual split logic if 'split_file' is provided
loader = GliomaDataLoader(DATA_PATH, split_file=SPLIT_FILE, memory_mapped=True)

Initializing Loader...
Loading data from ../data/processed/glioma_dataset.h5...
Data Loaded. Shape: (249783, 1738)
Unique Samples: 58
Sample ID Example: 'HF-448_V5B_1.h5_3' (Type: <class 'numpy.str_'>)


## 3. Run Benchmark (5-Fold CV)

In [ ]:
# --- 5-Fold Grouped Cross-Validation ---
# We iterate directly over the loader, which handles the splitting logic internally.

fold_results = []
feature_masks = []

print(f"🚀 Starting Training via Loader Iterator...")

# The loader yields a dictionary for each fold
for split_data in loader:
    fold_idx = split_data['fold']
    print(f"\n--- Fold {fold_idx+1} / 5 ---")
    
    # 1. Unpack Data from Loader
    # The loader returns tuples (X, y) for train and val
    X_train, y_train = split_data['train']
    X_val, y_val     = split_data['val']
    
    print(f"   Train Shape: {X_train.shape}, Val Shape: {X_val.shape}")
    
    # 2. Instantiate Apollo Pipeline 
    # (Re-instantiate per fold to ensure a fresh start)
    model = ApolloPipeline(n_features=20, random_state=42)
    
    # 3. Train
    # This runs: RF Selector -> Top 20 Features -> Scaler -> SVM
    model.fit(X_train, y_train)
    
    # 4. Save the Trained Model
    save_path = os.path.join(MODEL_DIR, f"apollo_fold_{fold_idx+1}.joblib")
    joblib.dump(model, save_path)
    print(f"   💾 Model saved to: {save_path}")
    
    # 5. Evaluate
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    
    # 6. Track Results
    fold_results.append({
        'fold': fold_idx + 1,
        'accuracy': acc,
        'y_true': y_val,
        'y_pred': y_pred
    })
    
    # 7. Track Selected Features
    feature_masks.append(model.selected_indices_)
    
    print(f"   ✅ Fold Accuracy: {acc:.4f}")

# --- Summary ---
mean_acc = np.mean([r['accuracy'] for r in fold_results])
print(f"\n🏆 Average Accuracy: {mean_acc:.4f}")

🚀 Starting Training via Loader Iterator...


TypeError: 'GliomaDataLoader' object is not iterable

## 4. Analysis & Reporting

In [ ]:
# Aggregate Accuracy
accuracies = [r['accuracy'] for r in fold_results]
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print(f"\n=== Final Results ===")
print(f"Apollo Baseline Accuracy: {mean_acc:.4f} (+/- {std_acc:.4f})")

# Save Baseline Metrics for future comparison
results_df = pd.DataFrame(fold_results)
results_df.to_csv("../results/baseline_metrics.csv", index=False)
print("Saved metrics to ../results/baseline_metrics.csv")

# Save Feature Masks (The 'Golden Features')
np.save("../results/apollo_feature_masks.npy", feature_masks)
print("Saved feature masks to ../results/apollo_feature_masks.npy")

### Confusion Matrix (Aggregated)

In [ ]:
all_true = np.concatenate([r['y_true'] for r in fold_results])
all_pred = np.concatenate([r['y_pred'] for r in fold_results])

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Aggregated Confusion Matrix (Acc: {mean_acc:.4f})")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

In [ ]:
# # Future Usage: To make predictions on new spectra, simply load the trained model and call predict. The pipeline will handle feature selection and scaling automatically.
# loaded_model = joblib.load("../models/baselines/apollo_fold_1.joblib")
# prediction = loaded_model.predict(new_spectrum) 
# # ^ This handles feature selection and scaling automatically!